Dense FP16
✔ Wanda + Quantization
✔ SLiM-LoRA + Quantization

# Datasets:
- Piqa - failed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Colab_Notebooks/ECS189G_Final_SLIM')
print(f"Current working directory: {os.getcwd()}")

In [ ]:
!pip install -r slim/requirements.txt

## Load tokens

In [ ]:


from google.colab import userdata
import os
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

if hf_token:
    hf_token = hf_token.strip()
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    # Do NOT set HF_HUB_DISABLE_IMPLICIT_TOKEN while you need auth
    os.environ.pop("HF_HUB_DISABLE_IMPLICIT_TOKEN", None)
    print("HF token set. Prefix:", hf_token[:10])
else:
    print("HF_TOKEN secret not found or empty.")

print("ENV HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))
print("ENV HUGGINGFACE_HUB_TOKEN present:", bool(os.environ.get("HUGGINGFACE_HUB_TOKEN")))
print("HF_HUB_DISABLE_IMPLICIT_TOKEN:", os.environ.get("HF_HUB_DISABLE_IMPLICIT_TOKEN"))

from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("meta-llama/Llama-2-13b-hf")
print("Loaded config OK:", cfg.model_type)




# Define Constants

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

MAIN_PATH = "/content/drive/MyDrive/Colab_Notebooks/ECS189G_Final_SLIM/slim/main_pile.py"
TASKS = ["arc_easy", "arc_challenge", "winogrande", "openbookqa"]
MODEL = "meta-llama/Llama-2-13b-hf"

In [ ]:
# NOT RUN THIS
import os, subprocess, textwrap


base_cmd = [
    "python", MAIN_PATH,
    "--model", MODEL,
    "--prune_method", "wanda",
    "--sparsity_ratio", "0",
    "--sparsity_type", "unstructured",
    "--lora_rank", "0",
    "--eval_dataset", "wikitext2",
    "--eval_batch_size", "1",
    "--test_lmharness",
    "--save_checkpoint_path", "checkpoints/dense_llama2_13b"
]

env = os.environ.copy()
failed_tasks = []
for t in TASKS:
    print("\n" + "="*80)
    print("RUNNING TASK:", t)
    print("="*80)

    cmd = base_cmd + [
        "--lm_harness_tasks", t,
        "--output_csv_path", f"results/llama2_13b_{t}.csv",
    ]

    p = subprocess.run(cmd, env=env)
    if p.returncode != 0:
        print("\n❌ FAILED on task:", t)
        failed_tasks.append(t)
if failed_tasks:
    print("\n❌ Failed tasks:", ", ".join(failed_tasks))
else:
    print("\n✅ All tasks completed successfully.")

In [ ]:
import os
import subprocess
from tqdm.auto import tqdm

RUN_TAG = f"wanda_u50_w4_i8_tile128_ig128_llama2_13b"
RESULTS_DIR = "results"
CKPT_DIR = "checkpoints"

BASE_CMD = [
    "python", MAIN_PATH,
    "--model", MODEL,
    "--prune_method", "wanda",
    "--sparsity_ratio", "0.5",
    "--sparsity_type", "unstructured",
    "--shift_zero_metrics",
    "--quantize_weight",
    "--bitwidth", "4",
    "--tiled_weight_quantization",
    "--weight_tile_size", "128",
    "--quantize_input",
    "--input_bitwidth", "8",
    "--input_group_size", "128",
    "--eval_dataset", "wikitext2",
    "--eval_batch_size", "1",
    "--test_lmharness",
    "--save_checkpoint_path", f"{CKPT_DIR}/{RUN_TAG}",
]

ENV = os.environ.copy()

def tail(text, n=50):
    """Return last n lines of text."""
    if not text:
        return "(no output)"
    lines = text.splitlines()
    return "\n".join(lines[-n:])


failed_tasks = []
completed = []

pbar = tqdm(TASKS, desc="LM Harness tasks", unit="task", dynamic_ncols=True)

for task in pbar:
    pbar.set_postfix_str(f"running={task}")

    out_csv = f"{RESULTS_DIR}/{RUN_TAG}_{task}.csv"

    cmd = BASE_CMD + [
        "--lm_harness_tasks", task,
        "--output_csv_path", out_csv,
    ]

    proc = subprocess.run(
        cmd,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if proc.returncode != 0:
        failed_tasks.append(task)

        tqdm.write(f"\n❌ FAILED: {task} (code={proc.returncode})")

        # Prefer stderr (actual errors), fallback to stdout
        err_text = proc.stderr if proc.stderr.strip() else proc.stdout

        tqdm.write("---- Last error output ----")
        tqdm.write(tail(err_text, 60))
        tqdm.write("---------------------------\n")

    else:
        completed.append(task)
        tqdm.write(f"✅ OK: {task} -> {out_csv}")


print("\n" + "=" * 80)
print("RUN SUMMARY")
print("=" * 80)
print("Model:", MODEL)
print("Run tag:", RUN_TAG)
print("Completed tasks:", ", ".join(completed) if completed else "(none)")
print("Failed tasks:", ", ".join(failed_tasks) if failed_tasks else "(none)")

In [ ]:
import os
import subprocess
from google.colab import userdata
from huggingface_hub import login, whoami
from tqdm.auto import tqdm

RUN_TAG = "wanda_u50_codeparrot_llama2_13b"
RESULTS_DIR = "results"
CKPT_DIR = "checkpoints"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

hf_token = userdata.get("HF_TOKEN").strip()

# Make this token the one used everywhere
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

login(token=hf_token)
print("Logged in as:", whoami())
print("Token prefix:", hf_token[:10])

# BASE_CMD = [
#     "python", MAIN_PATH,
#     "--model", MODEL,
#     "--hf_token", hf_token,
#     "--prune_method", "wanda",
#     "--calibration_dataset", "pile_dm_math",
#     "--nsamples", "128",
#     "--eval_dataset", "pile_dm_math",
#     "--eval_batch_size", "1",
#     "--sparsity_ratio", "0.5",
#     "--sparsity_type", "unstructured",
#     "--lora_rank", "0",
#     "--test_lmharness",
#     "--save_checkpoint_path", f"{CKPT_DIR}/wanda_pile_dm_math_llama2_13b"
# ]
BASE_CMD = [
    "python", MAIN_PATH,
    "--model", MODEL,
    "--hf_token", hf_token,
    "--prune_method", "wanda",
    "--calibration_dataset", "codeparrot",
    "--nsamples", "128",
    "--eval_dataset", "codeparrot",
    "--eval_batch_size", "1",
    "--sparsity_ratio", "0.5",
    "--sparsity_type", "unstructured",
    "--lora_rank", "0",
    "--test_lmharness",
    "--save_checkpoint_path", f"{CKPT_DIR}/wanda_codeparrot_llama2_13b"
]
ENV = os.environ.copy()

def tail(text, n=50):
    if not text:
        return "(no output)"
    lines = text.splitlines()
    return "\n".join(lines[-n:])

failed_tasks = []
completed = []

pbar = tqdm(TASKS, desc="LM Harness tasks", unit="task", dynamic_ncols=True)

for task in pbar:
    pbar.set_postfix_str(f"running={task}")
    out_csv = f"{RESULTS_DIR}/{RUN_TAG}_{task}.csv"

    cmd = BASE_CMD + [
        "--lm_harness_tasks", task,
        "--output_csv_path", out_csv,
    ]

    proc = subprocess.run(
        cmd,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if proc.returncode != 0:
        failed_tasks.append(task)
        tqdm.write(f"\nFAILED: {task} (code={proc.returncode})")
        err_text = proc.stderr if proc.stderr.strip() else proc.stdout
        tqdm.write("---- Last error output ----")
        tqdm.write(tail(err_text, 60))
        tqdm.write("---------------------------\n")
    else:
        completed.append(task)
        tqdm.write(f"OK: {task} -> {out_csv}")

print("\n" + "=" * 80)
print("RUN SUMMARY")
print("=" * 80)
print("Model:", MODEL)
print("Run tag:", RUN_TAG)
print("Completed tasks:", ", ".join(completed) if completed else "(none)")
print("Failed tasks:", ", ".join(failed_tasks) if failed_tasks else "(none)")

# SLIM-LORA

In [ ]:
import os
import subprocess
from tqdm.auto import tqdm
from google.colab import userdata
from huggingface_hub import login, whoami

# --- Your paths / model / tasks ---
MAIN_PATH = "/content/drive/MyDrive/Colab_Notebooks/ECS189G_Final_SLIM/slim/main_pile.py"
TASKS = ["winogrande", "openbookqa"]
MODEL = "meta-llama/Llama-2-13b-hf"

# --- Auth ---
hf_token = userdata.get("HF_TOKEN").strip()
login(token=hf_token)

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("Logged in as:", whoami())
print("Token prefix:", hf_token[:10])

# --- Configuration ---
RUN_TAG = "pile_dm_math_slimlora_llama2_13b"
RESULTS_DIR = "results"
CKPT_DIR = "checkpoints"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Base args for Pile + SLiM-LoRA ---
BASE_CMD = [
    "python", MAIN_PATH,
    "--model", MODEL,
    "--hf_token", hf_token,

    # Pruning
    "--prune_method", "wanda",
    "--sparsity_ratio", "0.5",
    "--sparsity_type", "unstructured",

    # Pile DM Math
    "--calibration_dataset", "codeparrot",
    "--nsamples", "128",
    "--eval_dataset", "codeparrot",
    "--eval_batch_size", "1",

    # SLiM-LoRA
    "--slim_lora",
    "--lora_rank", "0.1",
    "--separate_lora",
    "--quantize_lora",
    "--lora_tile_size", "128",

    # Quantization
    "--shift_zero_metrics",
    "--quantize_weight",
    "--bitwidth", "4",
    "--column_wise_grouping",
    "--slim_quant",

    # Evaluation
    "--test_lmharness",

    "--save_checkpoint_path", f"{CKPT_DIR}/{RUN_TAG}",
]

ENV = os.environ.copy()

def tail(text, n=60):
    if not text:
        return "(no output)"
    lines = text.splitlines()
    return "\n".join(lines[-n:])

failed_tasks = []
completed = []

pbar = tqdm(TASKS, desc="LM Harness tasks (Pile + SLiM-LoRA)", unit="task", dynamic_ncols=True)

for task in pbar:
    pbar.set_postfix_str(f"running={task}")

    out_csv = f"{RESULTS_DIR}/{RUN_TAG}_{task}.csv"

    cmd = BASE_CMD + [
        "--lm_harness_tasks", task,
        "--output_csv_path", out_csv,
    ]

    proc = subprocess.run(
        cmd,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if proc.returncode != 0:
        failed_tasks.append(task)
        tqdm.write(f"\n❌ FAILED: {task} (returncode={proc.returncode})")
        err_text = proc.stderr if proc.stderr.strip() else proc.stdout
        tqdm.write("---- Last error output ----")
        tqdm.write(tail(err_text, 80))
        tqdm.write("---------------------------\n")
    else:
        completed.append(task)
        tqdm.write(f"✅ OK: {task} -> {out_csv}")

print("\n" + "=" * 80)
print("RUN SUMMARY")
print("=" * 80)
print("Model:", MODEL)
print("Main path:", MAIN_PATH)
print("Run tag:", RUN_TAG)
print("Completed tasks:", ", ".join(completed) if completed else "(none)")
print("Failed tasks:", ", ".join(failed_tasks) if failed_tasks else "(none)")

## PLOTS



In [ ]:
# ======= Cell: Compare hardcoded SLiM-Quant vs Pile runs (without mmlu) =======

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = "/content/drive/MyDrive/Colab_Notebooks/ECS189G_Final_SLIM/results"
PLOTS_DIR = "/content/drive/MyDrive/Colab_Notebooks/ECS189G_Final_SLIM/plots"
TASKS = ["arc_easy", "arc_challenge", "winogrande", "openbookqa"]

MODEL_LABEL = "LLaMA-2 13B"
CALIBRATION_LABEL = "Codeparrot"
TARGET_SIZE = "13B"

os.makedirs(PLOTS_DIR, exist_ok=True)

slim_quant_manual = {
    "run_prefix": "manual_slim_quant_from_screenshot",
    "method": "SLiM-Quant",
    "model_size": "13B",
    "average": 0.555247,
    "arc_easy": 0.763047,
    "arc_challenge": 0.429181,
    "winogrande": 0.708761,
    "openbookqa": 0.320000,
}

def infer_task_from_filename(fname, tasks=TASKS):
    base = os.path.basename(fname)
    for t in tasks:
        if base.endswith(f"_{t}.csv") or base.endswith(f"{t}.csv"):
            return t
    return None

def infer_run_prefix(fname, task):
    base = os.path.basename(fname)
    if base.endswith(f"_{task}.csv"):
        return base[: -(len(task) + 5)]
    if base.endswith(f"{task}.csv"):
        return base[: -(len(task) + 4)]
    return os.path.splitext(base)[0]

def infer_method_label(run_prefix):
    rp = run_prefix.lower()

    if "wanda_u50_pile_dm_math_llama2_13b" in rp:
        return "Wanda + 4b (Pile)"
    if "pile_dm_math_slimlora_llama2_13b" in rp:
        return "SLiM-LoRA (Pile)"

    return "Other"

def infer_model_size(run_prefix):
    rp = run_prefix.lower()
    if "13b" in rp:
        return "13B"
    if "7b" in rp:
        return "7B"
    return "?"

def extract_accuracy_from_csv(df, task):
    if len(df) == 0:
        return np.nan

    row = df.iloc[0]

    if task in df.columns and pd.notna(row[task]):
        return float(row[task])

    if "average" in df.columns and pd.notna(row["average"]):
        return float(row["average"])

    for col in ["acc,none", "acc", "accuracy"]:
        if col in df.columns and pd.notna(row[col]):
            return float(row[col])

    return np.nan

def pct(x):
    return 100.0 * x if pd.notna(x) else np.nan


csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

csv_files = [
    f for f in csv_files
    if (
        "wanda_u50_pile_dm_math_llama2_13b" in os.path.basename(f).lower()
        or "pile_dm_math_slimlora_llama2_13b" in os.path.basename(f).lower()
    )
]

print("Filtered CSV files used:")
for f in csv_files:
    print(os.path.basename(f))

records = []
for f in csv_files:
    task = infer_task_from_filename(f, TASKS)
    if task is None:
        continue

    run_prefix = infer_run_prefix(f, task)
    df = pd.read_csv(f)
    acc = extract_accuracy_from_csv(df, task)

    method = infer_method_label(run_prefix)
    if method == "Other":
        continue

    records.append({
        "run_prefix": run_prefix,
        "method": method,
        "model_size": infer_model_size(run_prefix),
        "task": task,
        "acc": acc,
        "file": os.path.basename(f),
    })

long_df = pd.DataFrame(records)

if not long_df.empty:
    wide_from_csv = (
        long_df.pivot_table(
            index=["run_prefix", "method", "model_size"],
            columns="task",
            values="acc",
            aggfunc="first",
        )
        .reset_index()
    )
else:
    wide_from_csv = pd.DataFrame(columns=["run_prefix", "method", "model_size"] + TASKS)

task_cols = [t for t in TASKS if t in wide_from_csv.columns]
if task_cols:
    wide_from_csv["average"] = wide_from_csv[task_cols].mean(axis=1, skipna=True)
else:
    wide_from_csv["average"] = np.nan

manual_df = pd.DataFrame([slim_quant_manual])

all_cols = sorted(set(wide_from_csv.columns).union(set(manual_df.columns)))
wide_from_csv = wide_from_csv.reindex(columns=all_cols)
manual_df = manual_df.reindex(columns=all_cols)

wide = pd.concat([manual_df, wide_from_csv], ignore_index=True)

order = [
    "SLiM-Quant",
    "Wanda + 4b (codeparrot)",
    "SLiM-LoRA (codeparrot)",
]

wide["method_order"] = wide["method"].apply(lambda x: order.index(x) if x in order else 999)
wide = wide.sort_values(["method_order", "model_size", "run_prefix"]).drop(columns=["method_order"])

task_cols = [t for t in TASKS if t in wide.columns]

print("\nMerged runs found:")
display(wide[["run_prefix", "method", "model_size", "average"] + task_cols])

print("\nClean table:")
clean_table = wide[["method", "model_size", "average"] + task_cols].reset_index(drop=True)
display(clean_table)

# Plot 1
subset = wide[wide["model_size"] == TARGET_SIZE].copy()
if subset.empty:
    subset = wide.copy()

avg_by_method = subset.groupby("method")["average"].mean().map(pct)
avg_by_method = avg_by_method.reindex(order).dropna()

plt.figure(figsize=(9, 4.5))
bars = plt.bar(avg_by_method.index, avg_by_method.values)

plt.ylabel("Average accuracy (%)")
plt.title(f"{MODEL_LABEL} with {CALIBRATION_LABEL}: Average Zero-Shot Accuracy")
plt.ylim(0, max(100, np.nanmax(avg_by_method.values) + 10))
plt.xticks(rotation=15)

for bar, val in zip(bars, avg_by_method.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.5,
        f"{val:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "compare_avg_accuracy_no_mmlu.png"), dpi=300, bbox_inches="tight")
plt.show()

# Plot 2
subset = wide[wide["model_size"] == TARGET_SIZE].copy()
if subset.empty:
    subset = wide.copy()

subset["method_order"] = subset["method"].apply(lambda x: order.index(x) if x in order else 999)
subset = subset.sort_values("method_order").reset_index(drop=True)

methods_lbl = subset["method"].tolist()
vals_pct = (subset[task_cols] * 100.0).to_numpy()

plt.figure(figsize=(12, 5))
x = np.arange(len(task_cols))
bar_w = 0.8 / max(1, len(methods_lbl))

for i, method in enumerate(methods_lbl):
    plt.bar(x + i * bar_w, vals_pct[i], width=bar_w, label=method)

plt.xticks(x + (len(methods_lbl) - 1) * bar_w / 2, task_cols, rotation=0)
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title(f"{MODEL_LABEL} with {CALIBRATION_LABEL}: Per-Task Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "compare_per_task_accuracy_no_mmlu.png"), dpi=300, bbox_inches="tight")
plt.show()

# Plot 3
heat_df = clean_table.copy()
heat_df_display = heat_df.copy()

for col in ["average"] + task_cols:
    heat_df_display[col] = heat_df_display[col] * 100.0

fig, ax = plt.subplots(figsize=(13, max(2.5, 0.6 * len(heat_df_display))))
ax.axis("off")

table_data = []
for _, row in heat_df_display.iterrows():
    table_data.append(
        [row["method"], row["model_size"]]
        + [f"{row[col]:.2f}%" if pd.notna(row[col]) else "NA" for col in ["average"] + task_cols]
    )

col_labels = ["method", "model_size", "average"] + task_cols

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)

plt.title(f"{MODEL_LABEL} with {CALIBRATION_LABEL}: Results Table", pad=20)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "compare_results_table_no_mmlu.png"), dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved plot files:")
for f in sorted(glob.glob(os.path.join(PLOTS_DIR, "*no_mmlu*"))):
    print(f)

In [ ]:
print("Calibration dataset:", "codeparrot")
print("Eval dataset:", "codeparrot")
print("Run tag:", RUN_TAG)

Disable the runtime so not waster resources. Only run this overnight, or when you will leave for a long time

In [ ]:

from google.colab import runtime
runtime.unassign()